# 🌿 AGRO AI — Phase 2: train & evaluate our crop-health model (Google Colab Free)

**You only need to run the cells top to bottom.** Each cell says what it does in plain words.

### What happens here
1. **Download** three free photo collections: *PlantVillage* (clean lab photos of leaves), *PlantDoc* (real-world leaf photos) and *COCO* (everyday photos: phones, people, cars… used so the model learns "this is **not** a plant").
2. **Split** the photos into **train** (the model studies these), **validation** (we tune settings on these) and **test** (a final exam it never sees while learning). Photos of the *same leaf* always stay on the same side — otherwise the exam would be too easy.
3. **Train** a small model (MobileNetV3-Small, "transfer learning": it starts from a model that already understands images).
4. **Measure** accuracy, precision, recall, F1 and a confusion matrix — **separately** for PlantVillage, PlantDoc and COCO (we never blend them into one flattering number).
5. **Export** `agro_model.onnx` (a ~6 MB file) and download everything as one zip.

### Before you start
* `Runtime ▸ Change runtime type ▸ T4 GPU` (free).
* Have `agroai_ml_bundle.zip` ready (it is in `backend/ml/` on your Mac; recreate with `./make_colab_bundle.sh`).
* Training takes roughly **1–2 hours** on a free T4 (not yet timed on Colab — the first run will tell). If Colab disconnects, just re-run all cells: your progress is saved to Google Drive and training **resumes**.

## 1 · Check the GPU and connect Google Drive (so progress survives a disconnect)

In [ ]:
import os, subprocess
print(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout or "⚠️ NO GPU — go to Runtime > Change runtime type > T4 GPU")
from google.colab import drive
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/agroai_phase2'
os.makedirs(SAVE, exist_ok=True)
# Checkpoints and reports are written straight to Drive:
os.environ['AGRO_RUNS_DIR'] = f'{SAVE}/runs'
os.environ['AGRO_REPORTS_DIR'] = f'{SAVE}/reports'
RUN = 'colab'         # name of this training run (change it to keep several experiments side by side)
print('Saving to', SAVE)

## 2 · Upload the code bundle and install the few extra packages

In [ ]:
import zipfile, glob
os.makedirs('/content/ml', exist_ok=True)
if not os.path.exists('/content/ml/train.py'):
    from google.colab import files
    up = files.upload()            # choose agroai_ml_bundle.zip
    zipfile.ZipFile(list(up)[0]).extractall('/content/ml')
%cd /content/ml
!pip -q install pandas scikit-learn matplotlib onnx onnxruntime onnxscript remotezip certifi
import torch; print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())

## 3 · Download the datasets (≈ 4 GB, a few minutes)
Safe to re-run: files that already exist are skipped. These stay on Colab's temporary disk — **never** in your app or on Render.

In [ ]:
%%bash
set -e
cd /content/ml
mkdir -p data/plantvillage data/plantdoc data/coco
HF=https://huggingface.co/datasets/mohanty/PlantVillage/resolve/main
# PlantVillage (CC-BY-SA-3.0): 54k lab photos of leaves, 38 classes. leaf-map.json says which photos show the SAME leaf.
if [ ! -d data/plantvillage/color ]; then
  curl -L -s --retry 5 -C - -o data/plantvillage/data.zip $HF/data.zip
  curl -L -s --retry 5 -o data/plantvillage/leaf-map.json $HF/leaf_grouping/leaf-map.json
  python - <<'PY'
import zipfile
z = zipfile.ZipFile('data/plantvillage/data.zip')
z.extractall('data/plantvillage/x', [n for n in z.namelist() if n.startswith('raw/color/')])
PY
  mv data/plantvillage/x/raw/color data/plantvillage/color && rm -rf data/plantvillage/x data/plantvillage/data.zip
fi
# PlantDoc (CC-BY-4.0): real-world leaf photos scraped from the internet
[ -d data/plantdoc/repo ] || git clone --depth 1 -q https://github.com/pratikkayal/PlantDoc-Dataset data/plantdoc/repo
# COCO val2017: everyday photos (we remove any that contain plants) + a tiny annotation file
[ -f data/coco/val2017.zip ] || curl -L -s --retry 5 -C - -o data/coco/val2017.zip http://images.cocodataset.org/zips/val2017.zip
python - <<'PY'
import os
from remotezip import RemoteZip
if not os.path.exists('data/coco/annotations/instances_val2017.json'):
    with RemoteZip('http://images.cocodataset.org/annotations/annotations_trainval2017.zip') as z:
        z.extract('annotations/instances_val2017.json', 'data/coco')
PY
echo "downloads done"; du -sh data/*

## 4 · Prepare the data (leak-free split)
This builds `data/manifest.csv`: every photo, its label and whether it is for **train / val / test**. It also removes PlantDoc training photos that are near-copies of PlantDoc test photos, and keeps photos of the same leaf together. Takes ~3–6 minutes.

In [ ]:
import os
if not os.path.exists('data/manifest.csv'):
    !python prepare_data.py
else:
    print('manifest.csv already exists — skipping')

## 5 · Look at the data (class balance matters!)
Some classes have far fewer photos than others. Training compensates with *balanced sampling* (rare classes are shown more often), and we report **per-class** results so weak classes can't hide behind a high overall number.

In [ ]:
import pandas as pd
df = pd.read_csv('data/manifest.csv')
print(df.groupby(['source','split']).size().unstack(fill_value=0))
tr = df[(df.source=='plantvillage') & (df.role=='known') & (df.split=='train')]
tr.label.value_counts().sort_values().plot.barh(figsize=(8,9), title='PlantVillage training photos per class')

## 6 · Train 🏋️
* **Phase 1 (2 epochs):** only the new final layer learns.
* **Phase 2:** the whole network is fine-tuned gently. Training stops early if the validation score stops improving.
* The best epoch is saved. If Colab disconnects, re-run cells 1–2 and this cell — it **resumes**.

In [ ]:
!python train.py --name {RUN} --epochs 14 --samples 16000 --workers 2 --resume

## 7 · Training curve

In [ ]:
import json, pandas as pd
h = pd.DataFrame(json.load(open(f"{os.environ['AGRO_RUNS_DIR']}/{RUN}/history.json")))
print(h.round(4).to_string(index=False))
h.set_index('epoch')[['val_acc','val_macro_f1']].plot(figsize=(7,3.5), title='Validation (model selection only — NOT the final score)', ylim=(0,1))

## 8 · Final exam 🎓 — evaluate honestly
1. Run the model once over validation + test photos.
2. **Calibrate** using validation photos only: fit a "temperature" (so confidence numbers are trustworthy) and pick the **UNKNOWN** and **NO_PLANT** thresholds from data — not a made-up 80%.
3. **Report on the test photos**, one dataset at a time: PlantVillage, PlantDoc (real-world), COCO (non-plant), and unsupported plants.

In [ ]:
!python evaluate.py logits    --run {RUN} --workers 2
!python evaluate.py calibrate --run {RUN}
!python evaluate.py report    --run {RUN}

In [ ]:
import json
R = json.load(open(f"{os.environ['AGRO_REPORTS_DIR']}/{RUN}/metrics.json"))
A, B = R['A_plantvillage_test_known_classes'], R['B_plantdoc_test_real_world_known_classes']
def line(name, r): print(f"{name:34s} n={r['n']:5d}  accuracy={r['accuracy_raw_argmax']:.3f} (95% CI {r['accuracy_ci95']})  precision={r['precision_macro']:.3f}  recall={r['recall_macro']:.3f}  F1={r['f1_macro']:.3f}")
line('PlantVillage test (lab photos)', A); line('PlantDoc test (REAL-WORLD photos)', B)
print()
print('COCO test, non-plant photos      :', R['C_coco_test_non_plant'])
print('Real plants wrongly called NO_PLANT:', R['C_false_no_plant_rate_on_real_plant_test_images'])
for k, v in R['D_unknown_unsupported_plants'].items(): print('Unsupported plants -', k, ':', v)
print('\nweakest PlantVillage classes:', A['weakest_classes'])
print('weakest PlantDoc classes    :', B['weakest_classes'])
print('most common PlantVillage confusions:', A['top_confusions'][:5])
print('\nTarget >= 80% held-out accuracy ->  PlantVillage:', 'PASS' if A['accuracy_raw_argmax']>=0.8 else 'below target', '| PlantDoc real-world:', 'PASS' if B['accuracy_raw_argmax']>=0.8 else 'below target (expected: real-world photos are harder)')

In [ ]:
from IPython.display import Image, display
for n in ('plantvillage','plantdoc'):
    display(Image(f"{os.environ['AGRO_REPORTS_DIR']}/{RUN}/confusion_{n}.png"))

### If a score is below target
Don't lower the bar — investigate. In order: (1) look at *weakest classes* and *top confusions* above; (2) check the class counts in section 5; (3) train longer (`--epochs 20`) or with a larger sample per epoch (`--samples 24000`); (4) strengthen augmentation in `mldata.py`; (5) re-run sections 6–8 with a new `RUN` name and compare. Real-world (PlantDoc) accuracy is normally much lower than lab accuracy — that gap is real and is reported, not hidden.

## 9 · Export the small inference model (ONNX) and measure it

In [ ]:
!python export_onnx.py --run {RUN}
import json; print(json.load(open(f"{os.environ['AGRO_REPORTS_DIR']}/{RUN}/performance_export.json")))

## 10 · Download the result 📦
This creates **one zip** containing the model, its metadata, thresholds, metrics and confusion matrices. On your Mac run:
`cd backend/ml && python install_model.py ~/Downloads/agroai_phase2_results.zip`

In [ ]:
import shutil, glob
rep = f"{os.environ['AGRO_REPORTS_DIR']}/{RUN}"; runs = f"{os.environ['AGRO_RUNS_DIR']}/{RUN}"
os.makedirs('/content/out', exist_ok=True)
for f in glob.glob(f'{rep}/export/*'): shutil.copy(f, '/content/out/')
for f in glob.glob(f'{rep}/*.json') + glob.glob(f'{rep}/*.png'): shutil.copy(f, '/content/out/')
for f in ('thresholds.json','history.json','classes.json'): shutil.copy(f'{runs}/{f}', '/content/out/')
shutil.make_archive('/content/agroai_phase2_results', 'zip', '/content/out')
shutil.copy('/content/agroai_phase2_results.zip', SAVE)       # also keep a copy in Drive
from google.colab import files; files.download('/content/agroai_phase2_results.zip')